In [2]:
import sys
sys.path.append('../')

import numpy as np 
import pandas as pd 

from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score

from plotly import express as px

from tutoriales.utils import plot_confusion_matrix, get_artifact_filename

import os

from json import loads

from joblib import load, dump

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

In [3]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

In [4]:
# Las predicciones del modelo LGB se guardan directamente como joblib
lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_2__variables_completas.joblib'))

In [5]:
MODEL_NAME = '01 DistilBert'
MODEL_VERSION = '3.0'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)

bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

[I 2026-05-03 23:52:47,215] Using an existing study with name '01 DistilBert_3.0' instead of creating a new one.


In [6]:
# Cargar el modelo ResNet
MODEL_NAME_RESNET = '04 ResNet Augment'
MODEL_VERSION_RESNET = '1.0.0'

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/optuna_artifacts/db.sqlite3",
    study_name=f'{MODEL_NAME_RESNET}_{MODEL_VERSION_RESNET}',
    load_if_exists=True
)

[I 2026-05-03 23:52:54,596] Using an existing study with name '04 ResNet Augment_1.0.0' instead of creating a new one.


In [7]:
import sys, numpy.core
# Shim: permite cargar joblib guardados con numpy >= 2.0 en entornos con numpy 1.x
sys.modules.setdefault("numpy._core", numpy.core)
for _sub in ["numeric", "multiarray", "umath", "fromnumeric", "arrayprint", "strings"]:
    mod = getattr(numpy.core, _sub, numpy.core)
    sys.modules.setdefault(f"numpy._core.{_sub}", mod)


In [8]:
resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_04 ResNet Augment_1.0.0_1.joblib'))

In [9]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

In [10]:
# Unir ResNet al dataframe fusionado
merged_datasets = merged_datasets.merge(
    resnet_dataset[['PetID', 'pred']].rename({'pred': 'resnet_pred_score'}, axis=1),
    on='PetID', how='outer'
)

In [12]:
merged_datasets.head()

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score,resnet_pred_score
0,002230dea,"[0.1899978071714615, 0.19156674759358114, 0.39...",1,"[0.019768383, 0.29405907, 0.5350564, 0.1289099...","[-0.99180895, 0.7428905, 0.48204118, 0.1142346..."
1,0063f83c9,"[0.04151454188448956, 0.16817983051369315, 0.2...",1,"[0.08671505, 0.1929812, 0.20181805, 0.08142725...","[-1.3278729, 0.98171896, -0.028774094, 0.35766..."
2,0073c33d0,"[0.002237043583970307, 0.2811560426124426, 0.2...",3,"[0.005494275, 0.10332396, 0.52773774, 0.354337...","[-1.4208313, 0.7716534, 1.2558718, 0.40233982,..."
3,00bfa5da9,"[0.004355462179918298, 0.0802758115882416, 0.1...",4,"[0.0050450535, 0.004738874, 0.015713248, 0.016...","[-2.9708593, -0.9604822, 0.53046876, 0.9579238..."
4,00c19f4fa,"[0.1734469555180642, 0.07581099850411004, 0.45...",2,"[0.0037266116, 0.056987002, 0.62413824, 0.3035...","[-2.509712, 0.45661384, 0.80324507, 0.8857457,..."


In [13]:
merged_datasets.isnull().mean().mul(100).round(2).rename('% nulos')


PetID                0.00
lgb_pred_score       0.00
AdoptionSpeed        0.00
bert_pred_score      0.10
resnet_pred_score    2.27
Name: % nulos, dtype: float64

In [ ]:
# Limpiar nulos (rellenar con arrays de ceros si algún modelo no tiene predicción para un PetID)
merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['resnet_pred_score']]
merged_datasets['bert_pred_score']   = [np.zeros(5) if type(i) is float else i for i in merged_datasets['bert_pred_score']]
merged_datasets['lgb_pred_score']    = [np.zeros(5) if type(i) is float else i for i in merged_datasets['lgb_pred_score']]


In [18]:
# Optimización con Optuna para 3 pesos
def objective(trial):
    # Definir pesos para los tres modelos
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_bert = trial.suggest_float('w_bert', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    
    # Normalización
    total_w = w_lgb + w_bert + w_resnet
    
    # Cálculo vectorizado para mayor velocidad
    # Convertimos las columnas de scores en una matriz 3D o sumamos directamente
    lgb_scores = np.stack(merged_datasets['lgb_pred_score'].values)
    bert_scores = np.stack(merged_datasets['bert_pred_score'].values)
    resnet_scores = np.stack(merged_datasets['resnet_pred_score'].values)
    
    combined_scores = (
        (w_lgb / total_w) * lgb_scores + 
        (w_bert / total_w) * bert_scores + 
        (w_resnet / total_w) * resnet_scores
    )
    
    preds_final = np.argmax(combined_scores, axis=1)
    
    return cohen_kappa_score(merged_datasets['AdoptionSpeed'], preds_final, weights='quadratic')

In [19]:
# Ejecutar el estudio
STORAGE_URL = "sqlite:///../work/db.sqlite3"
study_blend = optuna.create_study(
    direction='maximize',
    storage=STORAGE_URL,
    study_name="Ensemble_LGB_BERT_ResNet",
    load_if_exists=True
)
study_blend.optimize(objective, n_trials=100)

[I 2026-05-04 00:00:19,722] Using an existing study with name 'Ensemble_LGB_BERT_ResNet' instead of creating a new one.
[I 2026-05-04 00:00:19,941] Trial 1 finished with value: 0.4005812298757302 and parameters: {'w_lgb': 0.5134286177894414, 'w_bert': 0.05020058816855166, 'w_resnet': 0.05295114669891998}. Best is trial 1 with value: 0.4005812298757302.
[I 2026-05-04 00:00:20,083] Trial 2 finished with value: 0.38446746321614944 and parameters: {'w_lgb': 0.563078040834624, 'w_bert': 0.7005259429242011, 'w_resnet': 0.29947888721657634}. Best is trial 1 with value: 0.4005812298757302.
[I 2026-05-04 00:00:20,227] Trial 3 finished with value: 0.3877334399477307 and parameters: {'w_lgb': 0.7874822790372383, 'w_bert': 0.9032946240102552, 'w_resnet': 0.5198440840267726}. Best is trial 1 with value: 0.4005812298757302.
[I 2026-05-04 00:00:20,361] Trial 4 finished with value: 0.3669059112998063 and parameters: {'w_lgb': 0.41739681890750513, 'w_bert': 0.869164054489633, 'w_resnet': 0.668709290409

In [20]:
# Resultados
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

print(f"Mejor Kappa: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")

Mejor Kappa: 0.4171
Pesos óptimos: {'w_lgb': 0.7223046572770836, 'w_bert': 0.28708516278023644, 'w_resnet': 0.296888150640109}


In [21]:
# Crear la columna de predicción final optimizada
merged_datasets['blend_pred_score'] = [
    (best_params['w_lgb'] / sum_best_w) * r['lgb_pred_score'] +
    (best_params['w_bert'] / sum_best_w) * r['bert_pred_score'] +
    (best_params['w_resnet'] / sum_best_w) * r['resnet_pred_score']
    for _, r in merged_datasets.iterrows()
]

In [22]:
from optuna.visualization import plot_param_importances

# Generar el gráfico de importancia de hiperparámetros (pesos)
fig = plot_param_importances(study_blend)

fig.update_layout(
    title="Importancia de los Modelos en el Ensamble",
    xaxis_title="Importancia relativa",
    yaxis_title="Modelo (Pesos)",
    template="plotly_white"
)

fig.show()


In [23]:
from optuna.visualization import plot_contour

In [24]:
fig_contour = plot_contour(study_blend, params=['w_lgb', 'w_bert', 'w_resnet'])

fig_contour.update_layout(
    title="Interacción de Pesos y Rendimiento (Kappa)",
    width=900,
    height=800
)

fig_contour.show()

In [25]:
STORAGE_URL = "sqlite:///../work/db.sqlite3"

study_blend = optuna.create_study(
    direction='maximize', 
    storage=STORAGE_URL, 
    study_name="Ensemble_LGB_BERT_ResNet",
    load_if_exists=True # Esto permite pausar y continuar la optimización luego
)

study_blend.optimize(objective, n_trials=100)

[I 2026-05-04 00:03:00,339] Using an existing study with name 'Ensemble_LGB_BERT_ResNet' instead of creating a new one.
[I 2026-05-04 00:03:00,622] Trial 101 finished with value: 0.3971182930421082 and parameters: {'w_lgb': 0.8485097159478074, 'w_bert': 0.5108945834036265, 'w_resnet': 0.22702912019979749}. Best is trial 15 with value: 0.41713945763333393.
[I 2026-05-04 00:03:00,747] Trial 102 finished with value: 0.41044504567704154 and parameters: {'w_lgb': 0.9576229142088807, 'w_bert': 0.23737576955815698, 'w_resnet': 0.25866689060385184}. Best is trial 15 with value: 0.41713945763333393.
[I 2026-05-04 00:03:00,891] Trial 103 finished with value: 0.4078482964819722 and parameters: {'w_lgb': 0.9796915111198391, 'w_bert': 0.27516457137695016, 'w_resnet': 0.28188242555021786}. Best is trial 15 with value: 0.41713945763333393.
[I 2026-05-04 00:03:01,041] Trial 104 finished with value: 0.4028281630593863 and parameters: {'w_lgb': 0.9284089977872968, 'w_bert': 0.26905655642431786, 'w_resne

In [ ]:
# Guardar el resultado final en un archivo temporal y subirlo a Optuna
final_filename = "merged_predictions_optimized.joblib"
dump(merged_datasets, final_filename)


In [ ]:
# Subir el archivo como artefacto del mejor trial
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)
upload_artifact(
    trial=study_blend.best_trial, 
    file_path=final_filename, 
    artifact_store=artifact_store
)


In [ ]:
#-------------------------------------------------------------

In [ ]:
#merged_datasets

In [ ]:
#merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['bert_pred_score'] for i,r in merged_datasets.iterrows()]

In [ ]:
#merged_datasets['lgb_pred_score']

In [ ]:
#merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
#merged_datasets['bert_pred'] = [r.argmax() for r in merged_datasets['bert_pred_score']]
#merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                      merged_datasets['lgb_pred'], 
#                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                    merged_datasets['lgb_pred'], 
#                                                                    weights='quadratic')))

In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                      merged_datasets['bert_pred'], 
#                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                   merged_datasets['bert_pred'], 
#                                                                    weights='quadratic')))



In [ ]:
#plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
#                     merged_datasets['blended_pred'], 
#                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
#                                                                    merged_datasets['blended_pred'], 
#                                                                    weights='quadratic')))
